# Ingest + train (Vertex AI Workbench / Colab)

Independent portfolio prototype using synthetic and public data. No Etihad internal data, systems or proprietary pricing logic are used.

1. Ingest `offer_log_train` into BigQuery `ancillary_lab` (training columns only).
2. Train XGBoost **P(buy | price, booking context)** with a decreasing constraint on price.
3. Export `new_model/model.joblib` for the Streamlit lab. Paid Vertex Prediction is not required.

Same scripts: `gcp/01_ingest_offer_log.py` and `gcp/02_train_new_model.py`.

In [ ]:
%pip install xgboost scikit-learn joblib pyarrow pandas google-cloud-bigquery google-cloud-storage --quiet

If this runtime is empty, clone the public repo first. Skip this cell when the notebook already sits inside the repo.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO = "https://github.com/Dureduran/EtihadPricingDemo.git"
if not (Path.cwd() / "new_model").exists() and not (Path.cwd() / "EtihadPricingDemo" / "new_model").exists():
    subprocess.check_call(["git", "clone", "--depth", "1", REPO])
    os.chdir("EtihadPricingDemo")
elif (Path.cwd() / "EtihadPricingDemo" / "new_model").exists():
    os.chdir("EtihadPricingDemo")

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT))
print("ROOT", ROOT)

Colab only: authenticate, then set your project. Vertex Workbench already has ADC.

In [ ]:
import os

try:
    from google.colab import auth

    auth.authenticate_user()
    os.environ["GCP_INGEST_BQ"] = "1"
    print("Colab auth OK")
except ImportError:
    print("Not Colab; using Workbench ADC or local parquet")

# os.environ["GCP_PROJECT"] = "YOUR_PROJECT_ID"
# os.environ["GCP_TRAIN_PATH"] = "gs://YOUR_BUCKET/offer_log_train.parquet"

In [ ]:
import importlib.util
from pathlib import Path
import sys

ROOT = Path.cwd() if (Path.cwd() / "new_model").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
spec = importlib.util.spec_from_file_location(
    "gcp_ingest_offer_log", ROOT / "gcp" / "01_ingest_offer_log.py"
)
ingest_mod = importlib.util.module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(ingest_mod)
print("INGEST")
print(ingest_mod.ingest())

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd() if (Path.cwd() / "new_model").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
train_path = ROOT / "gcp" / "02_train_new_model.py"
print("TRAIN")
namespace = {"__name__": "__main__", "__file__": str(train_path)}
with open(train_path, encoding="utf-8") as handle:
    exec(compile(handle.read(), str(train_path), "exec"), namespace)